In [1]:
"""
exp05 Step 02: Continuous Fractional Replay On The Validation Span (PROTOCOL.md §6-§9)

One continuous path over the 44 validation quarters of the exp02-exp04 schedule (2015-04-17 -> 2026-04-15); the account
holds w = min(1, max(floor, sigma_target / sigma_hat)) of its equity in SPY, rebalanced only beyond a 20-point dead band.
    1. CANDIDATE PATHS (after the fact, optimistic for the best): the 4 candidates, each replayed over the whole span; its
       return in each period minus buy-and-hold's = its period score (4 x 44 = 176 validation trials).
    2. SELECTION: mean period score over the last 4 periods; ties: higher floor, then higher target quantile (closest to
       buy-and-hold).
    3. PRIOR-ONLY PATH (the honest estimate): the candidate selected at period f governs period f + 1; one continuous
       path from period 1 to the end, with buy-and-hold, the constant exposure at the path's mean weight, 20 random
       circular shifts of the path's weights, the success criteria, the log-excess interval and the cost sensitivity.

Prerequisite: step 01's signal check passed. STOP (PROTOCOL.md §9) if the prior-only path does not beat buy-and-hold, the
constant exposure and the median random shift. No test window is evaluated: the bars are cut at the last validation
session before any simulation. RUN_MODE: "validation_only" only.
Outputs: store04_experiments/exp05_vol_scaled_exposure/step02_continuous_replay_data/.
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import os
import pandas as pd
import numpy as np
import time
# IMPORT PLOTLY
import plotly.graph_objects as go
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp05_vol_scaled_exposure import config as exp_config
# IMPORT DATA LOADING, EXECUTION AND DAILY FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT THE CONTINUOUS REPLAY FUNCTIONS
from so.core.continuous_replay import get_replay_period_pdf
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp05_vol_scaled_exposure.exposure import get_schedule_tuple, get_rule_candidate_list, get_weight_builder_func, run_candidate_exposure_dict, run_prior_only_exposure_dict
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_pdf, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [3]:
# DEFINE THE RUN MODE: ONLY "validation_only" EXISTS FOR THIS EXPERIMENT (NO TEST WINDOW IS EVALUATED)
RUN_MODE_STR = "validation_only"
assert RUN_MODE_STR == "validation_only"
# DEFINE THE CANDIDATE KEYS AND THE TIE-BREAKS OF THE SELECTION (PROTOCOL.md §6: HIGHER FLOOR, THEN HIGHER QUANTILE)
KEY_COL_STR_LIST = ["floor", "target_quantile"]
TIE_BREAK_LIST = [("floor", False), ("target_quantile", False)]
# DISPLAY THE SETTINGS
print(f"Run Mode:\t{RUN_MODE_STR}\tFloors:\t{exp_config.FLOOR_LIST}\tTarget quantiles:\t{exp_config.TARGET_QUANTILE_LIST}\tDead band:\t{exp_config.DEAD_BAND}")

Run Mode:	validation_only	Floors:	[0.3, 0.5]	Target quantiles:	[0.5, 0.75]	Dead band:	0.2


In [4]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step02_continuous_replay_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "validation")

Experiment:	exp05_vol_scaled_exposure	Protocol:	1.0	Config Hash:	b1763b5981
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp05_vol_scaled_exposure/step02_continuous_replay_data/
ℹ️ No earlier 'validation' entry of exp05_vol_scaled_exposure in the trial log (config hash b1763b5981).


In [5]:
# CALL FUNCTION TO READ THE MINUTE BARS AND BUILD THE SCHEDULE FROM THE FULL SESSION CALENDAR (SAME 44 QUARTERS AS exp02-exp04)
complete_ohlcv_pdf = get_complete_ohlcv_pdf()
fold_pdf, _ = get_schedule_tuple(sorted(complete_ohlcv_pdf["date"].unique()))
# DEFINE THE LAST VALIDATION SESSION AND CUT THE BARS THERE (NO LATER BAR IS GIVEN TO THE SIMULATOR)
replay_end_date = pd.to_datetime(fold_pdf["valid_end"].iloc[-1]).date()
assert replay_end_date < pd.Timestamp(exp_config.UNTOUCHED_START_DATE_STR).date()
replay_ohlcv_pdf = complete_ohlcv_pdf[complete_ohlcv_pdf["date"] <= replay_end_date].reset_index(drop=True)
del complete_ohlcv_pdf
# CALL FUNCTION TO CONVERT THE CUT DATA TO ARRAYS, BUILD THE DAILY TABLE AND THE REPLAY PERIODS
ohlcv_array_dict = get_ohlcv_array_dict(replay_ohlcv_pdf)
daily_pdf = get_daily_feature_pdf(ohlcv_array_dict)
period_pdf = get_replay_period_pdf(fold_pdf, ohlcv_array_dict["session_date_list"])
# CHECK THAT THE VALIDATION QUARTERS ARE THOSE OF exp03 (ITS SAVED SCHEDULE, IF PRESENT)
exp03_schedule_path_str = f"{paths.get_experiment_data_path_str('exp03_ath_exit', 'step02_walk_forward_data')}fold_schedule_data.csv"
if os.path.exists(exp03_schedule_path_str):
    exp03_fold_pdf = pd.read_csv(exp03_schedule_path_str)
    same_quarters_bool = exp03_fold_pdf["valid_start"].astype(str).tolist() == fold_pdf["valid_start"].astype(str).tolist() and exp03_fold_pdf["valid_end"].astype(str).tolist() == fold_pdf["valid_end"].astype(str).tolist()
    print(f"Same validation quarters as exp03:\t{same_quarters_bool}")
    assert same_quarters_bool
# SAVE THE SCHEDULE AND THE PERIODS
write_csv_file_to_path(fold_pdf, f"{output_path_str}fold_schedule_data.csv", "W", alert_in=False)
write_csv_file_to_path(period_pdf, f"{output_path_str}replay_period_data.csv", "W", alert_in=False)
# DISPLAY INFORMATION
print(f"Bars given to the simulator:\t{replay_ohlcv_pdf['date'].min()} -> {replay_ohlcv_pdf['date'].max()}\t({len(daily_pdf):,} sessions)")
print(f"Folds:\t{len(fold_pdf)}\tPeriods:\t{len(period_pdf)}\t{period_pdf['period_start'].iloc[0]} -> {period_pdf['period_end'].iloc[-1]}\t({int(period_pdf['session_count'].sum()):,} sessions)")
# PREVIEW DATAFRAME
period_pdf.head()

Same validation quarters as exp03:	True
Bars given to the simulator:	2005-01-03 -> 2026-04-15	(5,353 sessions)
Folds:	44	Periods:	44	2015-04-17 -> 2026-04-15	(2,765 sessions)


,period_id,period_start,period_end,session_count,valid_start,valid_end
0,0,2015-04-17,2015-07-16,63,2015-04-17,2015-07-16
1,1,2015-07-17,2015-10-14,63,2015-07-17,2015-10-16
2,2,2015-10-15,2016-01-15,64,2015-10-15,2016-01-14
3,3,2016-01-19,2016-04-15,62,2016-01-19,2016-04-15
4,4,2016-04-18,2016-07-14,62,2016-04-18,2016-07-15


In [6]:
"""
Candidate paths (AFTER THE FACT: the best of 4 chosen with hindsight is optimistic).
"""
# DEFINE THE CANDIDATES AND THE WEIGHT BUILDER (TARGETS FIXED PER PERIOD FROM THE EARLIER VALUES)
candidate_list = get_rule_candidate_list()
weight_builder_func = get_weight_builder_func(daily_pdf, period_pdf)
# CALL FUNCTION TO REPLAY EVERY CANDIDATE AND SCORE IT PER PERIOD
start_time = time.time()
candidate_replay_dict = run_candidate_exposure_dict(ohlcv_array_dict, period_pdf, candidate_list, weight_builder_func, exp_config.UNTOUCHED_START_DATE_STR, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT)
candidate_period_pdf, candidate_summary_pdf = candidate_replay_dict["candidate_period_pdf"], candidate_replay_dict["candidate_summary_pdf"]
print(f"Candidates replayed in {time.time() - start_time:.0f} s")
# LOG EVERY (CANDIDATE, PERIOD) AS A VALIDATION TRIAL (ONE WRITE)
validation_trial_pdf = log_trial_pdf(exp_config, "validation", candidate_period_pdf, KEY_COL_STR_LIST, [col for col in candidate_period_pdf.columns if col.startswith("valid_")],
                                     False, fold_pdf, note_str_in="exp05 step02 candidate path, one period of the continuous fractional replay")
print(f"Validation trials logged:\t{len(validation_trial_pdf)}")
# SAVE THE CANDIDATES
write_csv_file_to_path(candidate_period_pdf, f"{output_path_str}candidate_period_data.csv", "W", alert_in=False)
write_csv_file_to_path(candidate_summary_pdf, f"{output_path_str}candidate_summary_data.csv", "W", alert_in=False)
# PREVIEW DATAFRAME
candidate_summary_pdf[["candidate_idx", "floor", "target_quantile", "strategy_total_return", "buy_hold_total_return", "excess_total_return", "strategy_annualized_return",
                       "buy_hold_annualized_return", "mean_weight", "rebalance_count", "period_windows_beating_buy_hold", "log_excess_annualized_log_excess",
                       "log_excess_ci_low", "log_excess_ci_high", "strategy_sharpe_ratio", "buy_hold_sharpe_ratio", "strategy_max_drawdown", "buy_hold_max_drawdown"]].round(4)

	candidate 0 {'floor': 0.3, 'target_quantile': 0.5}: total 173.93% vs buy & hold 235.39% | mean weight 82.3% | rebalances 50
	candidate 1 {'floor': 0.3, 'target_quantile': 0.75}: total 185.70% vs buy & hold 235.39% | mean weight 93.2% | rebalances 27


	candidate 2 {'floor': 0.5, 'target_quantile': 0.5}: total 166.37% vs buy & hold 235.39% | mean weight 83.3% | rebalances 33
	candidate 3 {'floor': 0.5, 'target_quantile': 0.75}: total 187.88% vs buy & hold 235.39% | mean weight 93.3% | rebalances 27
Candidates replayed in 2 s


Validation trials logged:	176


,candidate_idx,floor,target_quantile,strategy_total_return,buy_hold_total_return,excess_total_return,strategy_annualized_return,buy_hold_annualized_return,mean_weight,rebalance_count,period_windows_beating_buy_hold,log_excess_annualized_log_excess,log_excess_ci_low,log_excess_ci_high,strategy_sharpe_ratio,buy_hold_sharpe_ratio,strategy_max_drawdown,buy_hold_max_drawdown
0,0,0.3,0.50,1.7393,2.3539,-0.6146,0.0962,0.1166,0.8235,50,9,-0.0183,-0.0413,0.0070,0.8167,0.7067,-0.1686,-0.342
1,1,0.3,0.75,1.8570,2.3539,-0.4969,0.1004,0.1166,0.9319,27,14,-0.0145,-0.0294,-0.0007,0.7274,0.7067,-0.2345,-0.342
2,2,0.5,0.50,1.6637,2.3539,-0.6902,0.0934,0.1166,0.8333,33,9,-0.0208,-0.0400,0.0014,0.7545,0.7067,-0.2243,-0.342
3,3,0.5,0.75,1.8788,2.3539,-0.4751,0.1012,0.1166,0.9326,27,14,-0.0138,-0.0277,-0.0007,0.7318,0.7067,-0.2345,-0.342


In [7]:
"""
Prior-only path (the honest validation estimate) with its baselines on the same sessions.
"""
# CALL FUNCTION TO SELECT, REPLAY THE PATH AND RUN THE BASELINES
start_time = time.time()
prior_dict = run_prior_only_exposure_dict(ohlcv_array_dict, period_pdf, candidate_period_pdf, candidate_list, weight_builder_func, exp_config.UNTOUCHED_START_DATE_STR,
                                          KEY_COL_STR_LIST, TIE_BREAK_LIST, exp_config.SELECTION_POOLED_QUARTER_COUNT, exp_config.RANDOM_RUN_COUNT,
                                          exp_config.RANDOM_SHIFT_MIN_SESSIONS, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT)
summary_dict, baseline_summary_dict, log_excess_1m_dict = prior_dict["summary_dict"], prior_dict["baseline_summary_dict"], prior_dict["log_excess_1m_dict"]
metric_dict = prior_dict["simulation_dict"]["metric_dict"]
print(f"Prior-only path and baselines in {time.time() - start_time:.0f} s")
# SAVE THE PATH
write_csv_file_to_path(prior_dict["selection_pdf"], f"{output_path_str}selection_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["baseline_pdf"], f"{output_path_str}prior_only_baseline_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["simulation_dict"]["daily_equity_pdf"], f"{output_path_str}prior_only_daily_equity_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["simulation_dict"]["trade_pdf"], f"{output_path_str}prior_only_trade_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["cost_sensitivity_pdf"], f"{output_path_str}prior_only_cost_sensitivity_data.csv", "W", alert_in=False)
write_csv_file_to_path(pd.DataFrame([{key: value for key, value in summary_dict.items() if key != "baselines_beaten"}]), f"{output_path_str}prior_only_summary_data.csv", "W", alert_in=False)
# DISPLAY THE PATH AGAINST BUY AND HOLD
print(f"\nPrior-only path:\t{summary_dict['first_date']} -> {summary_dict['last_date']}\t({summary_dict['session_count']:,} sessions, {summary_dict['year_count']:.2f} years)")
print(f"Total return:\tpath {summary_dict['strategy_total_return']:.2%} | buy & hold {summary_dict['buy_hold_total_return']:.2%}\t(annualized {summary_dict['strategy_annualized_return']:.2%} vs {summary_dict['buy_hold_annualized_return']:.2%})")
print(f"Mean weight in SPY:\t{metric_dict['mean_weight']:.1%}\tRebalances:\t{metric_dict['rebalance_count']}\tCost paid:\t{metric_dict['cost_paid']:.2f}")
print(f"Periods won:\t{summary_dict['period_windows_beating_buy_hold']} of {summary_dict['period_window_count']}\t(sign test p = {summary_dict['period_sign_test_p_value']:.3f})"
      f"\tmean period excess {summary_dict['period_mean_excess_return']:.2%} (SD {summary_dict['period_std_excess_return']:.2%}, t = {summary_dict['period_t_stat']:.2f})"
      f"\tMDE {summary_dict['period_mde_per_window']:.2%} per period ({summary_dict['period_mde_per_year']:.2%} per year)")
print(f"Annualized log excess:\t{summary_dict['log_excess_annualized_log_excess']:.4f}\t95% interval (6-month blocks) [{summary_dict['log_excess_ci_low']:.4f}, {summary_dict['log_excess_ci_high']:.4f}]"
      f"\t(1-month blocks [{log_excess_1m_dict['ci_low']:.4f}, {log_excess_1m_dict['ci_high']:.4f}])\tsupported: {summary_dict['log_excess_supported']}")
print(f"Sharpe:\tpath {summary_dict['strategy_sharpe_ratio']:.2f} | buy & hold {summary_dict['buy_hold_sharpe_ratio']:.2f}\tMax drawdown:\tpath {summary_dict['strategy_max_drawdown']:.2%} | buy & hold {summary_dict['buy_hold_max_drawdown']:.2%}")
print(f"PRIMARY (beats buy & hold):\t{summary_dict['primary_success']}\tComparable:\t{summary_dict['comparable_return']}\tSecondary Sharpe / drawdown:\t{summary_dict['secondary_sharpe']} / {summary_dict['secondary_drawdown']}")
print(f"Constant exposure at {baseline_summary_dict['constant_exposure_weight']:.1%}:\t{baseline_summary_dict['constant_exposure_total_return']:.2%}")
print(f"Random shifts (median of {exp_config.RANDOM_RUN_COUNT}):\t{baseline_summary_dict['random_shift_median']:.2%}\tpath beats {baseline_summary_dict['random_shift_path_beats_share']:.0%} of runs")
# APPLY THE STOPPING RULE (PROTOCOL.md §9)
information_bool = baseline_summary_dict["random_shift_path_beats_share"] > 0.5 and summary_dict["all_baselines_beaten"]
verdict_str = "CANDIDATE RESULT PENDING REVIEW" if summary_dict["primary_success"] and information_bool else "STOP"
print(f"\nInformation test passed:\t{information_bool}\tVERDICT:\t{verdict_str}")
# PREVIEW THE COST SENSITIVITY
prior_dict["cost_sensitivity_pdf"].round(4)

Prior-only path and baselines in 5 s

Prior-only path:	2015-07-17 -> 2026-04-15	(2,702 sessions, 10.72 years)
Total return:	path 165.69% | buy & hold 229.74%	(annualized 9.54% vs 11.77%)
Mean weight in SPY:	86.3%	Rebalances:	32	Cost paid:	78.71
Periods won:	9 of 43	(sign test p = 1.000)	mean period excess -0.61% (SD 2.39%, t = -1.66)	MDE 1.02% per period (4.07% per year)
Annualized log excess:	-0.0199	95% interval (6-month blocks) [-0.0433, 0.0008]	(1-month blocks [-0.0459, 0.0081])	supported: False
Sharpe:	path 0.73 | buy & hold 0.71	Max drawdown:	path -23.20% | buy & hold -34.21%
PRIMARY (beats buy & hold):	False	Comparable:	False	Secondary Sharpe / drawdown:	False / False
Constant exposure at 86.3%:	198.52%
Random shifts (median of 20):	192.15%	path beats 15% of runs

Information test passed:	False	VERDICT:	STOP


,slippage_per_share,total_return,buy_hold_return,excess_return
0,0.00,1.6583,2.2975,-0.6392
1,0.01,1.6569,2.2974,-0.6405
2,0.02,1.6565,2.2973,-0.6408


In [8]:
"""
Baselines of the prior-only path (same sessions, same costs, same dead band).
"""
# PREVIEW DATAFRAME
prior_dict["baseline_pdf"][["baseline", "shift_sessions", "total_return", "sharpe_ratio", "max_drawdown", "mean_weight", "rebalance_count"]].round(4)

,baseline,shift_sessions,total_return,sharpe_ratio,max_drawdown,mean_weight,rebalance_count
0,model,NaN,1.6569,0.7331,-0.2320,0.8634,32.0
1,buy_hold,NaN,2.2974,0.7083,-0.3421,NaN,NaN
2,constant_exposure,NaN,1.9852,0.7072,-0.3114,0.9087,0.0
3,random_shift_00,344.0,1.9198,0.7330,-0.2546,0.8753,33.0
4,random_shift_01,2022.0,2.1715,0.7891,-0.2490,0.8867,33.0
5,random_shift_02,1730.0,1.8403,0.6843,-0.3319,0.8578,30.0
6,random_shift_03,1201.0,2.1493,0.7411,-0.3031,0.8737,31.0
7,random_shift_04,1187.0,2.0807,0.7094,-0.3422,0.8848,35.0
8,random_shift_05,2230.0,1.5618,0.6109,-0.3251,0.8815,32.0
9,random_shift_06,336.0,1.7940,0.6991,-0.2455,0.8874,33.0


In [9]:
# DISPLAY THE SELECTED CANDIDATE PER PERIOD (THE CHOICE OF PERIOD f GOVERNS PERIOD f + 1)
selection_pdf = prior_dict["selection_pdf"]
print("Selections (candidate: periods):", selection_pdf.groupby(["floor", "target_quantile"]).size().to_dict())
selection_pdf[["fold_id", "floor", "target_quantile", "pooled_score", "pooled_quarter_count"]].tail(12).round(4)

Selections (candidate: periods): {(0.3, 0.5): 7, (0.5, 0.5): 9, (0.5, 0.75): 28}


,fold_id,floor,target_quantile,pooled_score,pooled_quarter_count
32,32,0.5,0.75,-0.0069,4
33,33,0.5,0.75,-0.0068,4
34,34,0.5,0.75,0.0000,4
35,35,0.5,0.75,0.0000,4
36,36,0.5,0.75,0.0000,4
37,37,0.5,0.75,0.0000,4
38,38,0.5,0.75,0.0000,4
39,39,0.3,0.50,0.0028,4
40,40,0.5,0.75,-0.0155,4
41,41,0.5,0.75,-0.0156,4


In [10]:
# CREATE A FIGURE OF THE PRIOR-ONLY PATH vs BUY AND HOLD (GROWTH OF 1) AND OF ITS WEIGHT
path_equity_pdf, buy_hold_equity_pdf = prior_dict["simulation_dict"]["daily_equity_pdf"], prior_dict["buy_hold_dict"]["daily_equity_pdf"]
fig = go.Figure()
fig.add_trace(go.Scatter(x=path_equity_pdf["date"], y=path_equity_pdf["equity"] / config.INITIAL_CAPITAL, name="prior-only path", line={"width": 1.5}))
fig.add_trace(go.Scatter(x=buy_hold_equity_pdf["date"], y=buy_hold_equity_pdf["equity"] / config.INITIAL_CAPITAL, name="buy & hold", line={"width": 1.5}))
fig.add_trace(go.Scatter(x=path_equity_pdf["date"], y=path_equity_pdf["weight"], name="weight in SPY", line={"width": 1}, yaxis="y2"))
fig.update_layout(template="plotly_dark", width=1100, height=450, title="exp05 prior-only path vs buy-and-hold (continuous replay, growth of 1)",
                  yaxis2={"overlaying": "y", "side": "right", "range": [0, 1.05], "title": "weight"})
fig.show()

In [11]:
# LOG ONE SUMMARY ENTRY OF THE RUN (THE CANDIDATE PERIODS ARE ALREADY LOGGED; THIS RECORDS THE HONEST ESTIMATE THAT WAS SEEN)
summary_trial_dict = log_trial_dict(exp_config, "summary", {"run_mode": RUN_MODE_STR, "selection": f"pooled {exp_config.SELECTION_POOLED_QUARTER_COUNT} periods", "path": "prior-only continuous fractional replay"},
                                    {**{key: value for key, value in summary_dict.items() if key != "baselines_beaten"}, **baseline_summary_dict,
                                     "log_excess_1m_ci_low": log_excess_1m_dict["ci_low"], "log_excess_1m_ci_high": log_excess_1m_dict["ci_high"],
                                     "mean_weight": metric_dict["mean_weight"], "rebalance_count": metric_dict["rebalance_count"], "verdict": verdict_str,
                                     "best_after_the_fact_total_return": float(candidate_summary_pdf["strategy_total_return"].max())},
                                    False, note_str_in="exp05 step02 validation summary (prior-only continuous fractional path, baselines, MDE, log-excess interval)")
print(f"Summary logged:\t{summary_trial_dict['trial_id']}")

Summary logged:	20261006143550262337_b1763b5981


In [12]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)

,experiment,stage,trial_count,config_count,test_touched_count,first_logged_ts,last_logged_ts
0,exp01_minute_entry,signal_check,2,2,0,2026-10-05T15:17:38,2026-10-06T12:46:46
1,exp01_minute_entry,summary,1,1,0,2026-10-05T16:16:30,2026-10-05T16:16:30
2,exp01_minute_entry,validation,540,1,0,2026-10-05T15:36:53,2026-10-05T16:16:18
3,exp02_stop_reentry,exploration,14,1,0,2026-10-05T14:47:50,2026-10-05T14:47:50
4,exp02_stop_reentry,signal_check,1,1,0,2026-10-05T14:48:41,2026-10-05T14:48:41
5,exp02_stop_reentry,summary,1,1,0,2026-10-05T14:50:29,2026-10-05T14:50:29
6,exp02_stop_reentry,validation,792,1,0,2026-10-05T14:49:18,2026-10-05T14:49:48
7,exp03_ath_exit,exploration,15,1,0,2026-10-05T14:55:04,2026-10-05T14:55:04
8,exp03_ath_exit,summary,1,1,0,2026-10-05T14:56:49,2026-10-05T14:56:49
9,exp03_ath_exit,validation,660,1,0,2026-10-05T14:55:55,2026-10-05T14:56:12
